# Quick SLM — 09 · Measurements the paper is still missing

Every open TODO in the v1 paper needs a number off Drive rather than an argument.
This notebook collects all of them in one pass and writes a single JSON.

| section | what is missing |
|---|---|
| 3.8 | bytes per token, per source |
| 5 | embedding norms for the reserved tokens |
| 6, 8 | wall-clock and peak memory |
| 7 | the training loss curve (Figure 2) |
| 8 | vocabulary-padding speedup, measured not assumed |
| 9.5 | the fine-tuning loss curve |

Each section is independent. If one fails, run the rest — the JSON records what
was collected and what was not, so a partial answer is still useful.

## 1. Mount and locate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, json, math
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), f'{framework_dir} not found'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)
from v1.quick_slm_trainer.paths import Layout
from v1.quick_slm_trainer.config import pretrain_v1, sft_v1

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
LOGS_DIR = DRIVE_ROOT / 'logs'
layout = Layout(drive_root=DRIVE_ROOT)
cfg = pretrain_v1()

M = {'collected': {}, 'failed': {}}
def record(key, value):
    M['collected'][key] = value
    print(f'  [ok] {key}')
def failed(key, why):
    M['failed'][key] = str(why)
    print(f'  [--] {key}: {why}')
print('ready')

## 2. Wall-clock and peak memory  (sections 6 and 8)

The trainer writes `tokens_per_sec` and `gpu_mem_gb` to `logs/train.jsonl` at
every logging interval. Wall-clock is taken from the first and last record's
timestamps if present, and otherwise reported as a projection and labelled one —
dividing the budget by a steady-state throughput is not a measurement.

In [ ]:
import collections

def read_jsonl(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows

TRAIN_LOG = LOGS_DIR / 'train.jsonl'
if not TRAIN_LOG.exists():
    failed('pretrain_log', f'{TRAIN_LOG} not found')
    rows = []
else:
    rows = read_jsonl(TRAIN_LOG)
    print(f'{len(rows):,} log records; keys seen: {sorted({k for r in rows for k in r})}')

if rows:
    def col(name):
        return [r[name] for r in rows if isinstance(r.get(name), (int, float))]

    mem, tps = col('gpu_mem_gb'), col('tokens_per_sec')
    if mem:
        record('peak_gpu_mem_gb', round(max(mem), 2))
    if tps:
        record('tokens_per_sec', {'median': round(sorted(tps)[len(tps)//2]),
                                  'max': round(max(tps)), 'n': len(tps)})
    ts = col('wall_time') or col('elapsed_s') or col('time')
    if len(ts) >= 2 and max(ts) > min(ts):
        record('wall_clock_hours_measured', round((max(ts) - min(ts)) / 3600, 2))
    elif tps:
        med = sorted(tps)[len(tps)//2]
        record('wall_clock_hours_projected',
               {'hours': round(cfg.run.target_tokens / med / 3600, 2),
                'note': 'PROJECTION from median throughput, not a measurement'})
    # The loss curve for Figure 2.
    curve = [{'step': r.get('step'), 'loss': r.get('loss'), 'lr': r.get('lr')}
             for r in rows if r.get('step') is not None and r.get('loss') is not None]
    if curve:
        record('pretrain_loss_curve', curve)
        print(f'   loss curve: {len(curve):,} points, '
              f'step {curve[0]["step"]} -> {curve[-1]["step"]}')

## 3. The fine-tuning loss curve  (section 9.5)

Same shape, from whichever log `05_sft_train` wrote. Table 14 has the endpoint;
this is the shape across 899 steps and three epochs.

In [ ]:
candidates = [LOGS_DIR / 'sft_train.jsonl', LOGS_DIR / 'sft.jsonl',
              layout.sft_dir / 'train.jsonl', LOGS_DIR / 'train_sft.jsonl']
found = next((p for p in candidates if p.exists()), None)
if found is None:
    failed('sft_loss_curve',
           'no SFT log at ' + ', '.join(str(p) for p in candidates) +
           ' -- check what 05 wrote and point this cell at it')
else:
    rows = read_jsonl(found)
    curve = [{'step': r.get('step'), 'loss': r.get('loss'), 'lr': r.get('lr'),
              'val_loss': r.get('val_loss')}
             for r in rows if r.get('step') is not None]
    record('sft_loss_curve', curve)
    print(f'   {found.name}: {len(curve):,} points')

## 4. Bytes per token, per source  (section 3.8)

Table 5 states the recipe in tokens. If the code slice is materially denser or
sparser in bytes than the prose slices, the shares do not mean what they appear
to mean. One pass over the per-source `.bin` files answers it.

In [ ]:
import numpy as np
from v1.quick_slm_trainer.tokenizer import load_tokenizer

try:
    tok = load_tokenizer(layout.tokenizer_dir, patch=False)
    SAMPLE = 2_000_000     # tokens per source; enough for a stable mean
    out = {}
    for key in cfg.data.budgets:
        path = layout.drive_source(key)
        if not path.exists():
            out[key] = None
            print(f'   {key:<14} missing at {path}')
            continue
        arr = np.memmap(path, dtype=np.uint16, mode='r')
        n = min(SAMPLE, len(arr))
        text = tok.decode(arr[:n].tolist())
        out[key] = {'tokens': int(n), 'bytes': len(text.encode('utf-8')),
                    'bytes_per_token': round(len(text.encode('utf-8')) / n, 3)}
        print(f'   {key:<14} {out[key]["bytes_per_token"]:.3f} bytes/token')
    record('bytes_per_token', out)
except Exception as e:
    failed('bytes_per_token', e)

## 5. Reserved-token embedding norms  (section 5)

Section 3.3 argues that a tied embedding row for a token that never appears in
the data still receives gradient on every step through the softmax denominator,
as a negative example, so its logit is driven down and the row does not stay near
initialisation. That is a claim about this checkpoint and it is checkable.

Reserved rows are compared against a random sample of ordinary rows from the same
matrix, since an absolute norm means nothing on its own.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

try:
    model = AutoModelForCausalLM.from_pretrained(str(layout.final_dir), dtype=torch.float32)
    W = model.get_input_embeddings().weight.detach()
    V = W.shape[0]
    print(f'   embedding matrix {tuple(W.shape)}')

    tok = load_tokenizer(layout.tokenizer_dir, patch=False)
    # The reserved tokens are the ones added past the base LLaMA-2 vocabulary.
    reserved_ids = list(range(32000, V))
    norms = W.norm(dim=1)
    g = torch.Generator().manual_seed(0)
    ordinary = torch.randperm(32000, generator=g)[:2000]

    res = {
        'n_reserved': len(reserved_ids),
        'reserved_mean_norm': round(norms[reserved_ids].mean().item(), 4),
        'reserved_min_norm': round(norms[reserved_ids].min().item(), 4),
        'reserved_max_norm': round(norms[reserved_ids].max().item(), 4),
        'ordinary_mean_norm': round(norms[ordinary].mean().item(), 4),
        'ordinary_median_norm': round(norms[ordinary].median().item(), 4),
        'per_token': {},
    }
    for i in reserved_ids:
        try:
            name = tok.convert_ids_to_tokens(i)
        except Exception:
            name = f'<id {i}>'
        res['per_token'][name] = round(norms[i].item(), 4)
    record('reserved_embedding_norms', res)
    print(f'   reserved mean {res["reserved_mean_norm"]}  vs ordinary mean {res["ordinary_mean_norm"]}')
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
except Exception as e:
    failed('reserved_embedding_norms', e)

## 6. Vocabulary padding, measured  (section 8)

Section 8 claims a speedup from padding the vocabulary to a multiple of 64 and
does not measure one. A hundred forward-backward steps at each size settles it.
Needs a GPU; skip if none is attached.

In [ ]:
RUN_VOCAB_BENCH = True

if not RUN_VOCAB_BENCH or not torch.cuda.is_available():
    failed('vocab_padding_speedup', 'skipped (needs a GPU)')
else:
    try:
        import time
        from v1.quick_slm_trainer.model import build_model

        def bench(vocab, steps=100, bs=4, ctx=1024):
            m = build_model(cfg, vocab_size=vocab).cuda().train()
            opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
            x = torch.randint(0, vocab, (bs, ctx), device='cuda')
            for _ in range(10):                       # warm up
                m(input_ids=x, labels=x).loss.backward(); opt.step(); opt.zero_grad()
            torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(steps):
                m(input_ids=x, labels=x).loss.backward(); opt.step(); opt.zero_grad()
            torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            del m, opt; torch.cuda.empty_cache()
            return dt

        unpadded, padded = bench(32_014), bench(32_128)
        record('vocab_padding_speedup', {
            'v_32014_seconds': round(unpadded, 3),
            'v_32128_seconds': round(padded, 3),
            'speedup': round(unpadded / padded, 4),
            'note': 'ratio > 1 means padding to 32,128 is faster',
        })
        print(f'   32,014: {unpadded:.2f}s   32,128: {padded:.2f}s   '
              f'speedup {unpadded/padded:.3f}x')
    except Exception as e:
        failed('vocab_padding_speedup', e)

## 7. Save

In [ ]:
out = LOGS_DIR / 'paper_measurements.json'
out.write_text(json.dumps(M, indent=2, default=str))
print(f'wrote {out}   ({out.stat().st_size/1e6:.2f} MB)')
print()
print('collected:', sorted(M['collected']))
print('failed   :', sorted(M['failed']))
for k, v in M['failed'].items():
    print(f'  {k}: {v}')